# 🫀 Sistem Prediksi Risiko Penyakit Jantung
**Dataset:** Heart Attack Prediction Indonesia  
**Target:** `heart_attack` (0 = Tidak, 1 = Ya)  

Notebook ini mencakup:
1. EDA & Preprocessing
2. Training Model Klasifikasi (Random Forest, Logistic Regression, XGBoost)
3. Evaluasi Model
4. Menyimpan model untuk deployment Streamlit

## 📦 1. Install & Import Library

In [ ]:
# Install library yang dibutuhkan
!pip install xgboost scikit-learn pandas numpy matplotlib seaborn joblib -q

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)
from xgboost import XGBClassifier

print('✅ Semua library berhasil diimport!')

## 📂 2. Load Dataset

In [ ]:
# Upload dataset ke Colab atau sesuaikan path
# from google.colab import files
# uploaded = files.upload()

df = pd.read_csv('heart_attack_prediction_indonesia.csv')
print(f'Shape dataset: {df.shape}')
df.head()

In [ ]:
print('=== INFO DATASET ===')
print(df.info())
print('\n=== MISSING VALUES ===')
print(df.isnull().sum())
print('\n=== DISTRIBUSI TARGET ===')
print(df['heart_attack'].value_counts())
print(f'\nRasio serangan jantung: {df["heart_attack"].mean()*100:.1f}%')

## 📊 3. Exploratory Data Analysis (EDA)

In [ ]:
# Distribusi variabel numerik
num_cols = ['age', 'blood_pressure_systolic', 'blood_pressure_diastolic',
            'fasting_blood_sugar', 'cholesterol_hdl', 'cholesterol_ldl',
            'triglycerides', 'waist_circumference', 'sleep_hours']

fig, axes = plt.subplots(3, 3, figsize=(15, 10))
for ax, col in zip(axes.flatten(), num_cols):
    df[col].hist(ax=ax, bins=30, color='steelblue', edgecolor='white')
    ax.set_title(col, fontsize=10)
    ax.set_xlabel('')
plt.suptitle('Distribusi Variabel Numerik', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Korelasi dengan target
plt.figure(figsize=(12, 6))
corr = df[num_cols + ['heart_attack']].corr()['heart_attack'].drop('heart_attack').sort_values()
corr.plot(kind='barh', color=['#e74c3c' if v > 0 else '#2ecc71' for v in corr.values])
plt.title('Korelasi Fitur Numerik terhadap Heart Attack', fontsize=13, fontweight='bold')
plt.axvline(0, color='black', linewidth=0.8)
plt.tight_layout()
plt.show()

In [ ]:
# Distribusi target per kategori penting
cat_cols = ['smoking_status', 'dietary_habits', 'stress_level', 'physical_activity']
fig, axes = plt.subplots(1, 4, figsize=(18, 5))
for ax, col in zip(axes, cat_cols):
    ct = df.groupby(col)['heart_attack'].mean() * 100
    ct.plot(kind='bar', ax=ax, color='#e74c3c', edgecolor='white')
    ax.set_title(f'% Heart Attack per\n{col}', fontsize=10)
    ax.set_ylabel('% Serangan Jantung')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=30)
plt.suptitle('Risiko Serangan Jantung per Kategori', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 🔧 4. Preprocessing

In [ ]:
# Definisikan kolom
CATEGORICAL_COLS = [
    'gender', 'region', 'income_level', 'smoking_status',
    'alcohol_consumption', 'physical_activity', 'dietary_habits',
    'air_pollution_exposure', 'stress_level', 'EKG_results'
]

FEATURE_COLS = [
    'age', 'gender', 'region', 'income_level', 'hypertension', 'diabetes',
    'cholesterol_level', 'obesity', 'waist_circumference', 'family_history',
    'smoking_status', 'alcohol_consumption', 'physical_activity', 'dietary_habits',
    'air_pollution_exposure', 'stress_level', 'sleep_hours',
    'blood_pressure_systolic', 'blood_pressure_diastolic', 'fasting_blood_sugar',
    'cholesterol_hdl', 'cholesterol_ldl', 'triglycerides', 'EKG_results',
    'previous_heart_disease', 'medication_usage'
]

TARGET_COL = 'heart_attack'

X = df[FEATURE_COLS].copy()
y = df[TARGET_COL].copy()

# Handle missing values
for col in X.select_dtypes(include='number').columns:
    X[col] = X[col].fillna(X[col].median())
for col in X.select_dtypes(include='object').columns:
    X[col] = X[col].fillna(X[col].mode()[0])

print(f'Missing values setelah impute: {X.isnull().sum().sum()}')

In [ ]:
# Label Encoding untuk kolom kategorikal
label_encoders = {}
for col in CATEGORICAL_COLS:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))
    label_encoders[col] = le

print('✅ Encoding selesai')
print(f'Shape X: {X.shape}')

# Simpan mapping encoding untuk dipakai di Streamlit
encoding_maps = {}
for col, le in label_encoders.items():
    encoding_maps[col] = dict(zip(le.classes_, le.transform(le.classes_)))
    print(f'  {col}: {encoding_maps[col]}')

In [ ]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f'Train size: {X_train.shape[0]:,}')
print(f'Test size : {X_test.shape[0]:,}')

## 🤖 5. Training Model

In [ ]:
# --- Logistic Regression ---
print('Training Logistic Regression...')
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_scaled, y_train)
y_pred_lr = lr.predict(X_test_scaled)
print('✅ Logistic Regression selesai')

In [ ]:
# --- Random Forest ---
print('Training Random Forest...')
rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
print('✅ Random Forest selesai')

In [ ]:
# --- XGBoost ---
print('Training XGBoost...')
xgb = XGBClassifier(
    n_estimators=200, max_depth=6, learning_rate=0.1,
    use_label_encoder=False, eval_metric='logloss',
    random_state=42, n_jobs=-1
)
xgb.fit(X_train, y_train)
y_pred_xgb = xgb.predict(X_test)
print('✅ XGBoost selesai')

## 📈 6. Evaluasi Model

In [ ]:
def evaluate_model(name, y_true, y_pred, y_prob=None):
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred)
    rec = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    auc = roc_auc_score(y_true, y_prob) if y_prob is not None else '-'
    print(f"\n{'='*40}")
    print(f"  Model: {name}")
    print(f"{'='*40}")
    print(f"  Accuracy : {acc:.4f}")
    print(f"  Precision: {prec:.4f}")
    print(f"  Recall   : {rec:.4f}")
    print(f"  F1-Score : {f1:.4f}")
    if y_prob is not None:
        print(f"  AUC-ROC  : {auc:.4f}")
    print(f"\n{classification_report(y_true, y_pred, target_names=['Tidak Sakit', 'Sakit Jantung'])}")
    return {'Model': name, 'Accuracy': acc, 'Precision': prec, 'Recall': rec, 'F1': f1, 'AUC': auc}

results = []
results.append(evaluate_model('Logistic Regression', y_test, y_pred_lr,
                              lr.predict_proba(X_test_scaled)[:,1]))
results.append(evaluate_model('Random Forest', y_test, y_pred_rf,
                              rf.predict_proba(X_test)[:,1]))
results.append(evaluate_model('XGBoost', y_test, y_pred_xgb,
                              xgb.predict_proba(X_test)[:,1]))

In [ ]:
# Tabel perbandingan model
df_results = pd.DataFrame(results)
print('\n📊 PERBANDINGAN MODEL:')
print(df_results.to_string(index=False))

# Visualisasi perbandingan
metrics = ['Accuracy', 'Precision', 'Recall', 'F1']
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(metrics))
width = 0.25
colors = ['#3498db', '#e74c3c', '#2ecc71']
for i, (_, row) in enumerate(df_results.iterrows()):
    vals = [row[m] for m in metrics]
    bars = ax.bar(x + i*width, vals, width, label=row['Model'], color=colors[i], alpha=0.85)
ax.set_xticks(x + width)
ax.set_xticklabels(metrics)
ax.set_ylim(0, 1.1)
ax.set_ylabel('Score')
ax.set_title('Perbandingan Performa Model', fontsize=13, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Confusion Matrix untuk model terbaik (Random Forest)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
models_preds = [
    ('Logistic Regression', y_pred_lr),
    ('Random Forest', y_pred_rf),
    ('XGBoost', y_pred_xgb)
]
for ax, (name, pred) in zip(axes, models_preds):
    cm = confusion_matrix(y_test, pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=['Tidak', 'Ya'])
    disp.plot(ax=ax, colorbar=False)
    ax.set_title(name, fontsize=11, fontweight='bold')
plt.suptitle('Confusion Matrix Semua Model', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Feature Importance (Random Forest)
feat_imp = pd.Series(rf.feature_importances_, index=FEATURE_COLS).sort_values(ascending=True)
plt.figure(figsize=(10, 8))
feat_imp.tail(20).plot(kind='barh', color='#e74c3c')
plt.title('Top 20 Feature Importance (Random Forest)', fontsize=13, fontweight='bold')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

## 💾 7. Simpan Model & Artifacts

In [ ]:
import os, json

os.makedirs('model_artifacts', exist_ok=True)

# Pilih model terbaik (XGBoost biasanya terbaik)
best_model = xgb  # ganti jika RF lebih baik

joblib.dump(best_model, 'model_artifacts/best_model.pkl')
joblib.dump(scaler, 'model_artifacts/scaler.pkl')
joblib.dump(label_encoders, 'model_artifacts/label_encoders.pkl')

# Simpan metadata
metadata = {
    'feature_cols': FEATURE_COLS,
    'categorical_cols': CATEGORICAL_COLS,
    'encoding_maps': encoding_maps
}
with open('model_artifacts/metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print('✅ Model dan artifacts berhasil disimpan!')
print('Files:', os.listdir('model_artifacts'))

In [ ]:
# Download artifacts (untuk upload ke GitHub/Streamlit)
import shutil
shutil.make_archive('model_artifacts', 'zip', 'model_artifacts')

# Uncomment untuk download di Google Colab
# from google.colab import files
# files.download('model_artifacts.zip')

print('✅ Zip berhasil dibuat: model_artifacts.zip')

## 🧪 8. Test Prediksi Manual

In [ ]:
# Contoh input pasien baru
sample_input = {
    'age': 55,
    'gender': 'Male',
    'region': 'Urban',
    'income_level': 'Middle',
    'hypertension': 1,
    'diabetes': 0,
    'cholesterol_level': 240,
    'obesity': 1,
    'waist_circumference': 95,
    'family_history': 1,
    'smoking_status': 'Current',
    'alcohol_consumption': 'Yes',
    'physical_activity': 'Low',
    'dietary_habits': 'Unhealthy',
    'air_pollution_exposure': 'High',
    'stress_level': 'High',
    'sleep_hours': 5.0,
    'blood_pressure_systolic': 145,
    'blood_pressure_diastolic': 95,
    'fasting_blood_sugar': 110,
    'cholesterol_hdl': 35,
    'cholesterol_ldl': 160,
    'triglycerides': 200,
    'EKG_results': 'Abnormal',
    'previous_heart_disease': 0,
    'medication_usage': 0
}

sample_df = pd.DataFrame([sample_input])
for col in CATEGORICAL_COLS:
    le = label_encoders[col]
    val = sample_df[col].astype(str).iloc[0]
    if val in le.classes_:
        sample_df[col] = le.transform([val])
    else:
        sample_df[col] = 0  # default jika tidak ditemukan

pred = best_model.predict(sample_df[FEATURE_COLS])[0]
prob = best_model.predict_proba(sample_df[FEATURE_COLS])[0][1]

print(f'\n🏥 HASIL PREDIKSI:')
print(f'   Diagnosa  : {"❗ BERISIKO SERANGAN JANTUNG" if pred == 1 else "✅ TIDAK BERISIKO"}')
print(f'   Probabilitas risiko: {prob*100:.1f}%')